# Multicollinearity analysis

In [1]:
import pandas as pd

In [2]:
# Show all rows and columns without truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_csv("../../Intermediate/09_combined_dataset.csv")
df.head()

TimeoutError: [Errno 60] Operation timed out

In [ ]:
# Check shape, dtypes, and missing values
print("DataFrame Shape:", df.shape)
print("DataFrame DTypes:")
print(df.dtypes)
print("Missing Values:")
print(df.isnull().sum())

DataFrame Shape: (17987952, 66)
DataFrame DTypes:
Unnamed: 0                    int64
time                            str
month                         int64
weekend                        bool
cell_start                      str
cell_end                        str
ride_count                    int64
all_day_jobs_start          float64
sex_female_start            float64
pay_1250_or_less_start      float64
job_ct_start                float64
pay_over_3333_start         float64
pay_1251_to_3333_start      float64
white_collar_jobs_start     float64
entertainment_jobs_start    float64
age_29_or_younger_start     float64
early_start_jobs_start      float64
all_day_jobs_end            float64
sex_female_end              float64
pay_1250_or_less_end        float64
job_ct_end                  float64
pay_over_3333_end           float64
pay_1251_to_3333_end        float64
white_collar_jobs_end       float64
entertainment_jobs_end      float64
age_29_or_younger_end       float64
early_start_jo

In [ ]:
# Check if NaN in park/school columns means "absence" rather than missing data
# by comparing with has_park flag
print(df[['has_park_start', 'park_name_start']].drop_duplicates().head(20))

      has_park_start        park_name_start
0                NaN                    NaN
4                1.0            Harris Park
9                1.0        Mosholu Parkway
27               1.0      Washington's Walk
39               1.0         St. James Park
60               1.0     Van Cortlandt Park
99               1.0         Riverside Park
159              1.0         Broadway Malls
378              1.0           Central Park
641              1.0      Park Avenue Malls
670              1.0       Carl Schurz Park
700              1.0   Riverside Park South
1047             1.0       Morningside Park
1135             1.0      St. Nicholas Park
1477             1.0     Wagner Houses Pool
1490             1.0     Marcus Garvey Park
1514             1.0      Harlem River Park
1549             1.0  Thomas Jefferson Park
1554             1.0        East River Walk
1627             1.0       Inwood Hill Park


In [ ]:
# Check if NaN in school columns also means "no school of this type" 
# by checking rows where all school_start columns are NaN
school_cols_start = ['high_school_start', 'secondary_school_start', 'early_schooling_start',
                      'elementary_school_start', 'k_12_school_start', 'k_8_school_start',
                      'middle_school_start', 'ungraded_school_start']

print(df[school_cols_start].notnull().sum(axis=1).value_counts())

0    11674071
8     6313881
Name: count, dtype: int64


In [ ]:
# Columns where NaN means "absence of feature"
# Convert to boolean flags: 1 if present, 0 if NaN
presence_cols = school_cols_start + [
    'high_school_end', 'secondary_school_end', 'early_schooling_end',
    'elementary_school_end', 'k_12_school_end', 'k_8_school_end',
    'middle_school_end', 'ungraded_school_end',
    'on_street_start', 'protected_lane_start',
    'on_street_end', 'protected_lane_end'
]

for col in presence_cols:
    df[col] = df[col].fillna(0)

# has_park is already a 1/NaN flag, just fill NaN with 0
df[['has_park_start', 'has_park_end']] = df[['has_park_start', 'has_park_end']].fillna(0).astype(int)

# Recheck missing values after conversion
print(df.isnull().sum().sort_values(ascending=False).head(20))

park_name_end             15961467
park_name_start           15945205
all_day_jobs_end              3139
early_start_jobs_end          3139
age_29_or_younger_end         3139
entertainment_jobs_end        3139
white_collar_jobs_end         3139
pay_1251_to_3333_end          3139
pay_over_3333_end             3139
job_ct_end                    3139
pay_1250_or_less_end          3139
sex_female_end                3139
res_lots_end                  2526
com_lots_end                  2526
manf_lots_end                 2526
female_pop_end                2525
age_15_to_34_end              2525
housing_units_end             2525
total_pop_end                 2525
pay_1250_or_less_start         496
dtype: int64


In [ ]:
# Drop park_name columns - not needed for regression, has_park flag is enough
df = df.drop(columns=['park_name_start', 'park_name_end'])

# Drop remaining rows with NaN (small fraction of demographic/job/zoning data)
print(f"Rows before dropping: {len(df)}")
df = df.dropna()
print(f"Rows after dropping: {len(df)}")

Rows before dropping: 17987952
Rows after dropping: 17984467


In [ ]:
# Compute correlation matrix for numeric feature columns
# Exclude identifier/categorical columns
exclude_cols = ['time', 'cell_start', 'cell_end', 'ride_count']
numeric_cols = [c for c in df.columns if c not in exclude_cols]

corr_matrix = df[numeric_cols].corr()

# Show pairs with high correlation (|corr| > 0.8), excluding self-correlation
high_corr = corr_matrix.where(
    abs(corr_matrix) > 0.8
)
high_corr_pairs = high_corr.stack().reset_index()
high_corr_pairs.columns = ['feature_1', 'feature_2', 'correlation']
high_corr_pairs = high_corr_pairs[high_corr_pairs['feature_1'] < high_corr_pairs['feature_2']]
high_corr_pairs = high_corr_pairs.sort_values('correlation', ascending=False)

# Drop NaN rows explicitly (pairs that didn't meet the >0.8 threshold)
high_corr_pairs = high_corr_pairs.dropna(subset=['correlation'])

print(high_corr_pairs)

                   feature_1            feature_2  correlation
1405     housing_units_start      total_pop_start     0.930758
1649       housing_units_end        total_pop_end     0.930722
917     pay_1250_or_less_end    pay_over_3333_end    -0.832138
307   pay_1250_or_less_start  pay_over_3333_start    -0.832209
487   pay_1251_to_3333_start  pay_over_3333_start    -0.921196
1097    pay_1251_to_3333_end    pay_over_3333_end    -0.921679


In [ ]:
df["time"].unique()

<ArrowStringArray>
['early_morning', 'morning', 'midday', 'evening', 'night']
Length: 5, dtype: str

In [ ]:
# Check dtype of weekend and month
print(df[['weekend', 'month']].dtypes)

# Check for zero-variance (constant) columns among numeric features
std_check = df[numeric_cols].std().sort_values()
print(std_check.head(20))

weekend     bool
month      int64
dtype: object
ungraded_school_start      0.030060
ungraded_school_end        0.030533
early_schooling_end        0.046820
early_schooling_start      0.047380
female_pop_end             0.049495
female_pop_start           0.049888
age_29_or_younger_start    0.056308
age_29_or_younger_end      0.056342
pay_1250_or_less_end       0.070580
pay_1250_or_less_start     0.070819
sex_female_end             0.077817
sex_female_start           0.078001
age_15_to_34_end           0.096804
age_15_to_34_start         0.097169
pay_1251_to_3333_end       0.100892
pay_1251_to_3333_start     0.100916
pay_over_3333_end          0.151723
pay_over_3333_start        0.151900
k_12_school_start          0.152361
k_12_school_end            0.152865
dtype: float64


In [ ]:
# Lower threshold to 0.7 to catch moderately strong correlations too
high_corr_07 = corr_matrix.where(
    (abs(corr_matrix) > 0.7)
)
high_corr_07_pairs = high_corr_07.stack().reset_index()
high_corr_07_pairs.columns = ['feature_1', 'feature_2', 'correlation']
high_corr_07_pairs = high_corr_07_pairs[high_corr_07_pairs['feature_1'] < high_corr_07_pairs['feature_2']]
high_corr_07_pairs = high_corr_07_pairs.dropna(subset=['correlation'])
high_corr_07_pairs = high_corr_07_pairs.sort_values('correlation', ascending=False)

print(high_corr_07_pairs)

                   feature_1            feature_2  correlation
1405     housing_units_start      total_pop_start     0.930758
1649       housing_units_end        total_pop_end     0.930722
917     pay_1250_or_less_end    pay_over_3333_end    -0.832138
307   pay_1250_or_less_start  pay_over_3333_start    -0.832209
487   pay_1251_to_3333_start  pay_over_3333_start    -0.921196
1097    pay_1251_to_3333_end    pay_over_3333_end    -0.921679


In [ ]:
# Drop the first element of each highly correlated pair
cols_to_drop = [
    'housing_units_start', 'housing_units_end',
    'pay_over_3333_start', 'pay_over_3333_end',
    'early_start_jobs_start', 'early_start_jobs_end' # dropping because it was constructed as a set of employment variables that all sum to 1
]

df = df.drop(columns=cols_to_drop)

print(df.shape)

(17984467, 58)


In [ ]:
df.head()

,Unnamed: 0,time,month,weekend,cell_start,cell_end,ride_count,all_day_jobs_start,sex_female_start,pay_1250_or_less_start,job_ct_start,pay_1251_to_3333_start,white_collar_jobs_start,entertainment_jobs_start,age_29_or_younger_start,all_day_jobs_end,sex_female_end,pay_1250_or_less_end,job_ct_end,pay_1251_to_3333_end,white_collar_jobs_end,entertainment_jobs_end,age_29_or_younger_end,age_15_to_34_start,total_pop_start,female_pop_start,age_15_to_34_end,total_pop_end,female_pop_end,high_school_start,secondary_school_start,early_schooling_start,elementary_school_start,k_12_school_start,k_8_school_start,middle_school_start,ungraded_school_start,high_school_end,secondary_school_end,early_schooling_end,elementary_school_end,k_12_school_end,k_8_school_end,middle_school_end,ungraded_school_end,on_street_start,protected_lane_start,on_street_end,protected_lane_end,has_park_start,has_park_end,res_lots_start,com_lots_start,manf_lots_start,res_lots_end,com_lots_end,manf_lots_end,grid_distance
0,0,early_morning,9,False,892a1001203ffff,892a1001213ffff,1,0.253927,0.479058,0.178010,76.4,0.252182,0.352531,0.185864,0.196335,0.364839,0.434783,0.060491,48.090909,0.247637,0.043478,0.209830,0.179584,0.294713,296.56,0.520232,0.280457,380.450000,0.524379,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0,75.0,20.0,1.0,67.0,13.0,0.0,1.0
1,1,early_morning,9,False,892a1001203ffff,892a100a88bffff,1,0.253927,0.479058,0.178010,76.4,0.252182,0.352531,0.185864,0.196335,0.088773,0.563969,0.052219,42.555556,0.229765,0.010444,0.143603,0.127937,0.294713,296.56,0.520232,0.306293,420.120000,0.542226,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0,75.0,20.0,1.0,80.0,0.0,0.0,12.0
2,2,early_morning,9,False,892a1001203ffff,892a100ac87ffff,1,0.253927,0.479058,0.178010,76.4,0.252182,0.352531,0.185864,0.196335,0.091410,0.503855,0.090859,113.500000,0.191079,0.247797,0.154736,0.178414,0.294713,296.56,0.520232,0.330548,202.956522,0.524207,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0,0,75.0,20.0,1.0,109.0,11.0,21.0,6.0
3,3,early_morning,9,False,892a1001203ffff,892a100ac8bffff,1,0.253927,0.479058,0.178010,76.4,0.252182,0.352531,0.185864,0.196335,0.556193,0.685498,0.194562,220.666667,0.353776,0.165861,0.251360,0.157704,0.294713,296.56,0.520232,0.317589,413.900000,0.492752,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0,75.0,20.0,1.0,50.0,55.0,0.0,5.0
4,4,early_morning,9,False,892a100120bffff,892a1001207ffff,1,0.002613,0.532811,0.199768,1148.0,0.211092,0.141405,0.031069,0.157085,0.150424,0.563559,0.055085,36.307692,0.186441,0.243644,0.110169,0.108051,0.000000,0.00,0.000000,0.281634,536.866667,0.535204,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1,0,5.0,1.0,2.0,167.0,0.0,0.0,2.0


In [ ]:
# One-hot encode 'time' column (5 categories: early_morning, morning, midday, evening, night)
df = pd.get_dummies(df, columns=['time'], prefix='time', drop_first=True)

print(df.shape)
print(df.columns[df.columns.str.startswith('time_')])

(17984467, 61)
Index(['time_evening', 'time_midday', 'time_morning', 'time_night'], dtype='str')


In [ ]:
# Group by season instead of month
season_map = {12: 'winter', 1: 'winter', 2: 'winter',
              3: 'spring', 4: 'spring', 5: 'spring',
              6: 'summer', 7: 'summer', 8: 'summer',
              9: 'fall', 10: 'fall', 11: 'fall'}

df['season'] = df['month'].map(season_map)
df = pd.get_dummies(df, columns=['season'], prefix='season', drop_first=True)
df = df.drop(columns=['month'])

In [ ]:
# Check cardinality of cell_start and cell_end
print(df['cell_start'].nunique())
print(df['cell_end'].nunique())

1577
1581


In [ ]:
print(len(df[df['cell_start']==df['cell_end']]))
df = df[df['cell_start']!=df['cell_end']]

253326


In [ ]:
# Check granularity using the one-hot encoded time and season columns instead
group_cols = ['cell_start', 'cell_end', 'weekend'] + list(df.columns[df.columns.str.startswith('season_')]) + list(df.columns[df.columns.str.startswith('time_')])

print(df.duplicated(subset=group_cols).sum())

In [ ]:
# Inspect a duplicated group to see what differs between rows
dup_mask = df.duplicated(subset=group_cols, keep=False)
sample_group = df[dup_mask].sort_values(group_cols).head(20)
print(sample_group[['cell_start', 'cell_end', 'weekend', 'ride_count']])

Empty DataFrame
Columns: [cell_start, cell_end, weekend, ride_count]
Index: []


In [ ]:
# Filter to one exact combination including a specific time bucket
one_group_exact = df[(df['cell_start'] == '892a100104bffff') & 
                      (df['cell_end'] == '892a100104bffff') & 
                      (df['season_spring'] == 1) & 
                      (df['weekend'] == False) &
                      (df['time_morning'] == 1)]

print(len(one_group_exact))
varying_cols = one_group_exact.nunique()
print(varying_cols[varying_cols > 1])

0
Series([], dtype: int64)


In [ ]:
# Now that we've confirmed only ride_count varies within a group, aggregate
agg_cols = [c for c in df.columns if c not in group_cols + ['ride_count']]
df_grouped = df.groupby(group_cols, as_index=False).agg(
    {**{'ride_count': 'sum'}, **{c: 'first' for c in agg_cols}}
)
df = df_grouped

print(df.shape)

In [ ]:
# Keep cell_start/cell_end as identifiers, but exclude from regression features
id_cols = ['cell_start', 'cell_end']
feature_cols = [c for c in df.columns if c not in id_cols + ['ride_count']]

print(f"Number of feature columns: {len(feature_cols)}")
print(feature_cols)

Number of feature columns: 59
['weekend', 'season_spring', 'season_summer', 'season_winter', 'time_evening', 'time_midday', 'time_morning', 'time_night', 'all_day_jobs_start', 'sex_female_start', 'pay_1250_or_less_start', 'job_ct_start', 'pay_1251_to_3333_start', 'white_collar_jobs_start', 'entertainment_jobs_start', 'age_29_or_younger_start', 'all_day_jobs_end', 'sex_female_end', 'pay_1250_or_less_end', 'job_ct_end', 'pay_1251_to_3333_end', 'white_collar_jobs_end', 'entertainment_jobs_end', 'age_29_or_younger_end', 'age_15_to_34_start', 'total_pop_start', 'female_pop_start', 'age_15_to_34_end', 'total_pop_end', 'female_pop_end', 'high_school_start', 'secondary_school_start', 'early_schooling_start', 'elementary_school_start', 'k_12_school_start', 'k_8_school_start', 'middle_school_start', 'ungraded_school_start', 'high_school_end', 'secondary_school_end', 'early_schooling_end', 'elementary_school_end', 'k_12_school_end', 'k_8_school_end', 'middle_school_end', 'ungraded_school_end', 'o

In [ ]:
# Verify all feature columns are numeric
print(df[feature_cols].dtypes.value_counts())

float64    49
bool        8
int64       2
Name: count, dtype: int64


In [ ]:
df.head(10)

,cell_start,cell_end,weekend,season_spring,season_summer,season_winter,time_evening,time_midday,time_morning,time_night,ride_count,all_day_jobs_start,sex_female_start,pay_1250_or_less_start,job_ct_start,pay_1251_to_3333_start,white_collar_jobs_start,entertainment_jobs_start,age_29_or_younger_start,all_day_jobs_end,sex_female_end,pay_1250_or_less_end,job_ct_end,pay_1251_to_3333_end,white_collar_jobs_end,entertainment_jobs_end,age_29_or_younger_end,age_15_to_34_start,total_pop_start,female_pop_start,age_15_to_34_end,total_pop_end,female_pop_end,high_school_start,secondary_school_start,early_schooling_start,elementary_school_start,k_12_school_start,k_8_school_start,middle_school_start,ungraded_school_start,high_school_end,secondary_school_end,early_schooling_end,elementary_school_end,k_12_school_end,k_8_school_end,middle_school_end,ungraded_school_end,on_street_start,protected_lane_start,on_street_end,protected_lane_end,has_park_start,has_park_end,res_lots_start,com_lots_start,manf_lots_start,res_lots_end,com_lots_end,manf_lots_end,grid_distance
0,892a100104bffff,892a1001203ffff,False,False,True,False,False,True,False,False,1,0.473684,0.507368,0.176842,43.181818,0.309474,0.008421,0.484211,0.250526,0.253927,0.479058,0.178010,76.400000,0.252182,0.352531,0.185864,0.196335,0.281564,477.333333,0.516201,0.294713,296.560000,0.520232,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0,133.0,16.0,0.0,75.0,20.0,1.0,4.0
1,892a100104bffff,892a1001207ffff,False,False,True,False,False,True,False,False,1,0.473684,0.507368,0.176842,43.181818,0.309474,0.008421,0.484211,0.250526,0.150424,0.563559,0.055085,36.307692,0.186441,0.243644,0.110169,0.108051,0.281564,477.333333,0.516201,0.281634,536.866667,0.535204,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0,133.0,16.0,0.0,167.0,0.0,0.0,4.0
2,892a100104bffff,892a1001207ffff,False,False,True,False,True,False,False,False,1,0.473684,0.507368,0.176842,43.181818,0.309474,0.008421,0.484211,0.250526,0.150424,0.563559,0.055085,36.307692,0.186441,0.243644,0.110169,0.108051,0.281564,477.333333,0.516201,0.281634,536.866667,0.535204,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0,133.0,16.0,0.0,167.0,0.0,0.0,4.0
3,892a100104bffff,892a1001207ffff,False,True,False,False,False,False,False,False,1,0.473684,0.507368,0.176842,43.181818,0.309474,0.008421,0.484211,0.250526,0.150424,0.563559,0.055085,36.307692,0.186441,0.243644,0.110169,0.108051,0.281564,477.333333,0.516201,0.281634,536.866667,0.535204,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0,133.0,16.0,0.0,167.0,0.0,0.0,4.0
4,892a100104bffff,892a1001207ffff,False,True,False,False,False,True,False,False,3,0.473684,0.507368,0.176842,43.181818,0.309474,0.008421,0.484211,0.250526,0.150424,0.563559,0.055085,36.307692,0.186441,0.243644,0.110169,0.108051,0.281564,477.333333,0.516201,0.281634,536.866667,0.535204,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0,133.0,16.0,0.0,167.0,0.0,0.0,4.0
5,892a100104bffff,892a100120bffff,False,False,True,False,False,False,True,False,2,0.473684,0.507368,0.176842,43.181818,0.309474,0.008421,0.484211,0.250526,0.002613,0.532811,0.199768,1148.000000,0.211092,0.141405,0.031069,0.157085,0.281564,477.333333,0.516201,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,1,133.0,16.0,0.0,5.0,1.0,2.0,5.0
6,892a100104bffff,892a100120bffff,False,False,True,False,False,True,False,False,1,0.473684,0.507368,0.176842,43.181818,0.309474,0.008421,0.484211,0.250526,0.002613,0.532811,0.199768,1148.000000,0.211092,0.141405,0.031069,0.157085,0.281564,477.333333,0.516201,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,1,133.0,16.0,0.0,5.0,1.0,2.0,5.0
7,892a100104bffff,892a100120bffff,False,True,False,False,False,False,True,False,2,0.473684,0.507368,0.176842,43.181818,0.30

In [ ]:
# Save cleaned dataset for regression modeling
df.to_csv("../../Intermediate/10_regression_ready_dataset.csv", index=False)

print("Saved successfully")
print(df.shape)

Saved successfully
(6991003, 62)
